### Etapa 4: Leitura dos arquivos 

In [63]:
import pandas as pd

df1 = pd.read_csv(
    "jan21_ago25.csv",
    sep=";",
    encoding="latin1",
    skiprows=9,
    decimal=","
)

df2 = pd.read_csv(
    "dados_A839_D_2025-09-01_2026-04-06.csv",
    sep=";",
    encoding="latin1",
    skiprows=9,
    decimal=","
)

df_crimes = pd.read_csv(
    "CrimesTratados.csv",
    sep=",", 
    encoding="latin1", 
    decimal=","
)




### 4.1 LIMPEZA INICIAL

In [64]:
df_meteo = pd.concat([df1, df2], ignore_index=True) 

In [65]:
# remover coluna vazia extra
df_meteo = df_meteo.loc[:, ~df_meteo.columns.str.contains("^Unnamed")]

# limpar nomes das colunas
df_meteo.columns = df_meteo.columns.str.strip()

# renomear colunas
df_meteo = df_meteo.rename(columns={
    "Data Medicao": "data_fato",
    "PRECIPITACAO TOTAL, DIARIO (AUT)(mm)": "precipitacao",
    "TEMPERATURA MAXIMA, DIARIA (AUT)(°C)": "temp_max",
    "TEMPERATURA MINIMA, DIARIA (AUT)(°C)": "temp_min",
    "UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)": "umidade",
    "VENTO, VELOCIDADE MEDIA DIARIA (AUT)(m/s)": "vento"
})

In [66]:
df_meteo.tail()

,data_fato,precipitacao,"TEMPERATURA MAXIMA, DIARIA (AUT)(Â°C)","TEMPERATURA MINIMA, DIARIA (AUT)(Â°C)",umidade,vento
1922,2026-04-02,1.6,30,17.7,78.2,3.1
1923,2026-04-03,0,31.5,18.3,76.2,3.1
1924,2026-04-04,.8,30.3,18.3,74.3,2.3
1925,2026-04-05,0,26.5,19.3,81.3,4.3
1926,2026-04-06,0,30.3,17.7,74.7,4.9


In [70]:
df_meteo.rename(columns={"data": "data_fato"}, inplace=True)

# converter datas
df_meteo["data_fato"] = pd.to_datetime(df_meteo["data_fato"], errors="coerce")
df_crimes["data_fato"] = pd.to_datetime(df_crimes["data_fato"], errors="coerce")


# adicionar cidade
df_meteo["municipio_fato"] = "Passo Fundo"



In [71]:
df_meteo["data_fato"] = pd.to_datetime(df_meteo["data_fato"], format="%d/%m/%Y", errors="coerce")

df_crimes.head()


,sequencia,data_fato,hora_fato,grupo_fato,tipo_enquadramento,tipo_fato,municipio_fato,local_fato,bairro,quantidade_vitimas,idade_vitima,sexo_vitima,cor_vitima
0,1.0,2021-10-01,00:01:00,CRIMES,INCENDIO DOLOSO,CONSUMADO,NOVA ALVORADA,RESIDENCIA,JARDIM BELA VISTA,1.0,20.0,MASCULINO,BRANCA
1,2.0,2021-10-01,00:01:00,CRIMES,VIOLENCIA PSICOLOGICA CONTRA MULHER ART 147B,CONSUMADO,SANTA CRUZ DO SUL,RESIDENCIA,SANTA VITÃRIA,1.0,66.0,FEMININO,BRANCA
2,3.0,2021-10-01,00:01:00,CRIMES,FURTO SIMPLES EM RESIDENCIA,CONSUMADO,SANTA ROSA,RESIDENCIA,CRUZEIRO,1.0,49.0,FEMININO,BRANCA
3,4.0,2021-10-01,00:01:00,CRIMES,IMPORTUNACAO SEXUAL,CONSUMADO,RIO GRANDE,VIA PUBLICA,VILA SANTA TEREZA,1.0,42.0,FEMININO,BRANCA
4,5.0,2021-10-01,00:01:00,CRIMES,OUTRAS FRAUDES,CONSUMADO,PORTO ALEGRE,VIA PUBLICA,AZENHA,1.0,51.0,FEMININO,PARDA


In [72]:
df_final = pd.merge(
    df_crimes,
    df_meteo,
    on=["data_fato", "municipio_fato"],
    how="inner"
)


In [73]:
df_final.dropna(inplace=True)
df_final.isnull().sum()

sequencia                                0
data_fato                                0
hora_fato                                0
grupo_fato                               0
tipo_enquadramento                       0
tipo_fato                                0
municipio_fato                           0
local_fato                               0
bairro                                   0
quantidade_vitimas                       0
idade_vitima                             0
sexo_vitima                              0
cor_vitima                               0
precipitacao                             0
TEMPERATURA MAXIMA, DIARIA (AUT)(Â°C)    0
TEMPERATURA MINIMA, DIARIA (AUT)(Â°C)    0
umidade                                  0
vento                                    0
dtype: int64

In [74]:
df_final.to_csv("df_final.csv", index=False, encoding="latin1")